# 02 変数作成・EM/IM分析

`01_prepare_data.ipynb` が作った年別CSVから、3変数とEM・IMを続けて計算・保存します。
対象年は `settings.py` を確認し、このノートブックを上から順に実行してください。

- 前半A：日本の輸出額・世界の市場規模・日本の品目に限定した市場規模。
- 後半B：EM・IM、年別・全期間の集計。
- 保存先：`settings.py` の `VARIABLE_DIR`。最初に見る結果は `margins.csv`。

年別CSVがすでにあれば、このノートブックだけで分析できます。APIキーは不要です。


# A. 変数の作成

`01_prepare_data.ipynb` が作った年ごとファイル（`data/by_year_allp/trade_{年}.csv.gz`）から、
分析用の変数を組み立てる。

元データは **HS 1〜24類（食料・農水産物）× 2000〜2025年（標準設定） × 全報告国 × 全相手国 × 輸出**、
金額は `primaryValue`（USD、輸出なので FOB 建て）。

## 変数一覧

| 変数名 | 定義 | 単位 | 粒度 |
|---|---|---|---|
| `japancountry` | **日本から輸入国 *m* への輸出額の合計**。日本を報告国、*m* を相手国とする行を、HS 1〜24類の全品目について合算したもの | USD | 年 × 相手国 *m* |
| `worldcountry` | **世界各国から輸入国 *m* への総輸出額**。全報告国を対象に、*m* を相手国とする行を HS 1〜24類の全品目について合算したもの。*m* の輸入市場規模にあたり、`japancountry` はこの一部 | USD | 年 × 相手国 *m* |
| `japanx` | **日本が *m* へ輸出している品目に限定した、世界各国から *m* への輸出額**。その年に日本が *m* へ売っている HS6桁の集合を特定し、その品目について全報告国 → *m* を合算したもの。日本が競合している到達可能な市場規模。定義上 `japancountry ≤ japanx ≤ worldcountry` | USD | 年 × 相手国 *m* |

いずれも `partnerDesc == "World"`（全世界合計の行）は除外して集計する。
残すと個別相手国の合計と二重計上になるため。

各変数は3つの形で保存する。

| 形 | ファイル | 使いどころ |
|---|---|---|
| 年ごとに1ファイル | `{変数名}/{変数名}_{年}.csv` | その年だけ扱いたいとき |
| 横持ち（相手国 × 年） | `{変数名}_wide.csv` | 国ごとの推移を横に並べて見たいとき |
| 縦持ち（年 × 相手国） | `{変数名}.csv` | 回帰などパネル分析にそのまま使う |

変数を増やすときは末尾の雛形を複製する。

## A-1. セットアップ

In [ ]:
from pathlib import Path

import pandas as pd

from settings import YEARS, YEAR_DIR, VARIABLE_DIR

SRC = YEAR_DIR
OUT = VARIABLE_DIR

OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

files = {y: SRC / f"trade_{y}.csv.gz" for y in YEARS}
lack  = [y for y, p in files.items() if not p.exists()]
if lack:
    raise FileNotFoundError(
        f"年ごとファイルが無い: {lack}\n"
        "→ 01_prepare_data.ipynbを実行して作成すること")
print(f"入力: {SRC.resolve()}  ({len(files)} 年分)")

## A-2. 元データの読み込み

年別CSVを10万行ずつ読み、日本の明細と相手国×品目の市場集計を保持します。
同じデータを後半のEM・IM計算にも再利用します。


In [ ]:
JAPAN = 392        # 日本の reporterCode

from lib.trade_processing import summarize_trade, read_reporter

# 年別CSVの読み込みはここで1回。世界の明細は相手国×品目に集約して保持する。
japan_rows, markets_by_year = summarize_trade(files, reporter_code=JAPAN)


def load_reporter(reporter_code):
    """追加分析用。日本は読み込み済みのデータを再利用する。"""
    if reporter_code == JAPAN:
        return japan_rows.copy()
    return read_reporter(files, reporter_code)


print(f"日本の行数: {len(japan_rows):,}")
print(f"年        : {sorted(japan_rows['refYear'].unique())}")
print(f"フロー    : {sorted(japan_rows['flowDesc'].unique())}")
print(f"相手国数  : {japan_rows['partnerCode'].nunique()}（World 含む）")

## A-3. 変数① `japancountry`

**日本から輸入国 *m* への輸出額の合計**（HS 1〜24類を合算）。

- 単位は USD（名目）。`primaryValue` は輸出なので FOB 建て。
- `partnerDesc == "World"` は**全世界合計の行**なので除外する。
  残さないと「個別国の合計」と「World」が二重に入る。
- 年ごとに1行 × 相手国。パネルデータとして使える形にする。

In [ ]:
VALUE = "primaryValue"     # 輸出額（USD）

# World（全世界合計）を除いた、実在する相手国のみ
japan_partner_rows = japan_rows[japan_rows["partnerDesc"] != "World"]

japancountry = (japan_partner_rows
                .groupby(["refYear", "partnerCode", "partnerDesc"], as_index=False)[VALUE]
                .sum()
                .rename(columns={"refYear": "year", VALUE: "japancountry"})
                .sort_values(["year", "japancountry"], ascending=[True, False])
                .reset_index(drop=True))

print(f"{len(japancountry):,} 行（年 × 相手国）")
japancountry.head(10)

### 検算

除外した `World` 行の合計と、相手国別の合計が一致するはず。
ズレる場合は、報告国が World 行だけを出して内訳を出していない年がある等の理由が考えられる。

In [ ]:
world = (japan_rows[japan_rows["partnerDesc"] == "World"]
         .groupby("refYear", as_index=False)[VALUE].sum()
         .rename(columns={"refYear": "year", VALUE: "world_total"}))
detail = japancountry.groupby("year", as_index=False)["japancountry"].sum()

chk = world.merge(detail, on="year")
chk["差"]     = chk["world_total"] - chk["japancountry"]
chk["差の率"] = (chk["差"] / chk["world_total"] * 100).round(2)
chk

## A-4. 確認

In [ ]:
# 上位10か国（直近年）
last = japancountry["year"].max()
top = japancountry[japancountry["year"] == last].nlargest(10, "japancountry").copy()
top["百万USD"] = (top["japancountry"] / 1e6).round(1)
print(f"{last}年 日本の農水産物・食品（HS1〜24類）輸出先 上位10")
top[["partnerDesc", "japancountry", "百万USD"]].reset_index(drop=True)

In [ ]:
# 主要国の推移（横持ちに変換）
wide = japancountry.pivot(index="year", columns="partnerDesc", values="japancountry")
cols = (japancountry.groupby("partnerDesc")["japancountry"].sum()
        .nlargest(8).index.tolist())
(wide[cols] / 1e6).round(1)      # 単位: 百万USD

## A-5. 保存 — 年ごとに分けて出力

同じ変数を用途に応じて3つの形で出す。

| 形 | ファイル | 使いどころ |
|---|---|---|
| **年ごとに1ファイル** | `japancountry/japancountry_{年}.csv` | その年だけ扱いたいとき |
| **横持ち（相手国 × 年）** | `japancountry_wide.csv` | 国ごとの推移を横に並べて見たいとき |
| 縦持ち（年 × 相手国） | `japancountry.csv` | 回帰などパネル分析にそのまま使う |

In [ ]:
# ① 年ごとに1ファイル
YEAR_DIR = OUT / "japancountry"
YEAR_DIR.mkdir(parents=True, exist_ok=True)

for y, g in japancountry.groupby("year"):
    p = YEAR_DIR / f"japancountry_{y}.csv"
    (g.drop(columns="year")
      .sort_values("japancountry", ascending=False)
      .reset_index(drop=True)
      .to_csv(p, index=False, encoding="utf-8-sig"))
    print(f"{y}: {len(g):>3} か国 → {p.name}")

print(f"\n{len(list(YEAR_DIR.glob('*.csv')))} ファイル: {YEAR_DIR.resolve()}")

In [ ]:
# ② 横持ち（相手国 × 年）— 国ごとの推移が1行で読める
japancountry_wide = (japancountry
                     .pivot(index=["partnerCode", "partnerDesc"],
                            columns="year", values="japancountry")
                     .reset_index())
japancountry_wide.columns.name = None

p = OUT / "japancountry_wide.csv"
japancountry_wide.to_csv(p, index=False, encoding="utf-8-sig")
print("保存:", p.resolve(), f"({len(japancountry_wide)} か国 × {len(YEARS)} 年)")

# 日本→中国の例（欠損は「その年に報告がない」を意味する）
japancountry_wide[japancountry_wide["partnerDesc"] == "China"]

In [ ]:
# ③ 縦持ち（パネル形式）
p = OUT / "japancountry.csv"
japancountry.to_csv(p, index=False, encoding="utf-8-sig")
print("保存:", p.resolve(), f"({len(japancountry):,} 行)")

## A-6. 変数② `worldcountry`

**世界各国から輸入国 *m* への総輸出額**（全報告国 × HS 1〜24類を合算）。年 × 相手国。

`japancountry` が「日本 → m」なのに対し、こちらは「世界全体 → m」。
つまり m の（この品目群における）**輸入市場の規模**にあたる。

### 集計の前提

- **全報告国を合算する。** 日本も含む。日本を除いた「日本以外の世界」が必要な場合は
  下のセルの `EXCLUDE_REPORTER` に `392` を入れる。
- `partnerDesc == "World"` の行は除外する（全世界合計なので m ではない）。
- 地域集計による二重計上は起きない。参照表で `isGroup=True` の報告国のうち
  データに現れるのは `Other Asia, nes`（490）だけで、これは台湾を指す個別の報告経済であり、
  他の報告国を束ねた集計ではない。`ASEAN` / `European Union` は報告国として出現しない。
- `reporterCode == partnerCode`（自国向け）の行は存在しない。

### 注意 — これは「m の輸入額」そのものではない

各国が申告した**輸出額（FOB）の積み上げ**なので、m 自身が申告する輸入額（CIF）とは一致しない。
運賃・保険料の差、報告漏れ、計上時期のズレがあるため、一般に輸入額のほうが大きくなる。

In [ ]:
EXCLUDE_REPORTER = None      # 例: 392 とすると「日本を除く世界」になる

# 標準設定は読み込み済みの市場集計を再利用する。
# 報告国を除く追加分析では、指定条件で再集計する。
world_markets = markets_by_year
if EXCLUDE_REPORTER is not None:
    _, world_markets = summarize_trade(files, exclude_reporter=EXCLUDE_REPORTER)
country_totals_by_year = []
for year_market in world_markets.values():
    country_totals = year_market.groupby(
        ["refYear", "partnerCode", "partnerDesc"], as_index=False
    )[VALUE].sum()
    country_totals_by_year.append(country_totals)

worldcountry = (pd.concat(country_totals_by_year, ignore_index=True)
                .groupby(["refYear", "partnerCode", "partnerDesc"], as_index=False)[VALUE].sum()
                .rename(columns={"refYear": "year", VALUE: "worldcountry"})
                .sort_values(["year", "worldcountry"], ascending=[True, False])
                .reset_index(drop=True))

print(f"\n{len(worldcountry):,} 行（年 × 相手国）")
worldcountry.head(10)

### 検算

`japancountry`（日本 → m）は `worldcountry`（世界 → m）の一部なので、
**すべての (年, m) で `japancountry` ≤ `worldcountry`** が成り立つはず。

In [ ]:
chk2 = japancountry.merge(worldcountry, on=["year", "partnerCode", "partnerDesc"], how="left")
bad = chk2[chk2["japancountry"] > chk2["worldcountry"] + 1e-6]
print("japancountry > worldcountry となる行:", len(bad), " ← 0 であること")

chk2["日本シェア%"] = (chk2["japancountry"] / chk2["worldcountry"] * 100).round(2)
print()
print("年ごとの世界合計と日本シェア:")
g = chk2.groupby("year").agg(世界=("worldcountry", "sum"), 日本=("japancountry", "sum"))
g["日本シェア%"] = (g["日本"] / g["世界"] * 100).round(3)
(g / [1e9, 1e9, 1]).round(3).rename(columns={"世界": "世界(10億USD)", "日本": "日本(10億USD)"})

In [ ]:
# 保存（japancountry と同じ3形式）
VAR = "worldcountry"
tbl = worldcountry

# ① 年ごと
d = OUT / VAR
d.mkdir(parents=True, exist_ok=True)
for y, g in tbl.groupby("year"):
    (g.drop(columns="year").sort_values(VAR, ascending=False).reset_index(drop=True)
      .to_csv(d / f"{VAR}_{y}.csv", index=False, encoding="utf-8-sig"))
print(f"① 年ごと {len(list(d.glob('*.csv')))} ファイル: {d.resolve()}")

# ② 横持ち（相手国 × 年）
wide = tbl.pivot(index=["partnerCode", "partnerDesc"], columns="year", values=VAR).reset_index()
wide.columns.name = None
wide.to_csv(OUT / f"{VAR}_wide.csv", index=False, encoding="utf-8-sig")
print(f"② 横持ち: {VAR}_wide.csv ({len(wide)} か国)")

# ③ 縦持ち
tbl.to_csv(OUT / f"{VAR}.csv", index=False, encoding="utf-8-sig")
print(f"③ 縦持ち: {VAR}.csv ({len(tbl):,} 行)")

wide[wide["partnerDesc"] == "China"]

## A-7. 変数③ `japanx`

**日本が輸入国 *m* へ輸出している品目に限定した、世界各国から *m* への輸出額**。

`worldcountry` は m の輸入市場を全品目で測るが、その中には日本が扱っていない品目も含まれる。
`japanx` は**日本が実際に m へ売っている品目だけ**に絞って世界の輸出額を合計するので、
日本が競合している市場の規模、つまり**到達可能な市場規模**を表す。

### 手順

1. **品目集合を特定** — その年に日本が m へ輸出している HS6桁コードの集合 *S(年, m)* を作る。
2. **世界の輸出額を合計** — 全報告国について、相手国が m かつ品目が *S(年, m)* に含まれる行を合算する。

品目集合は**年ごと・相手国ごとに別々**に作る。日本の品目構成は年で変わり、
相手国によっても違うため。全期間で固定した品目バスケットが必要な場合は、
下のセルの `FIXED_BASKET` を `True` にする。

### 大小関係

定義上、必ず次が成り立つ。

```
japancountry  ≤  japanx  ≤  worldcountry
（日本→m）      （世界→m、        （世界→m、
                 日本の品目のみ）    全品目）
```

日本の行には金額0・欠損がなく、`(相手国, 品目)` の組も一意なので、
集合の定義に曖昧さはない（2022年で6,933組）。

In [ ]:
FIXED_BASKET = False    # True: 全期間共通 / False: 年ごとの品目集合

partner_commodity_columns = ["partnerCode", "cmdCode"]
if FIXED_BASKET:
    fixed_basket = japan_partner_rows[partner_commodity_columns].drop_duplicates()

yearly_japanx = []
for year in YEARS:
    year_market = markets_by_year[year]  # 読み込み済みの年×相手国×品目市場

    # 1. 日本が輸出している「相手国×品目」の組を選ぶ。
    if FIXED_BASKET:
        exported_commodities = fixed_basket
    else:
        is_target_year = japan_partner_rows["refYear"].astype(str) == year
        year_japan_rows = japan_partner_rows.loc[is_target_year]
        exported_commodities = year_japan_rows[partner_commodity_columns].drop_duplicates()

    # 2. その品目に対応する世界の輸出額を取り出す。
    covered_market = year_market.merge(
        exported_commodities, on=partner_commodity_columns, how="inner"
    )

    # 3. 相手国ごとに市場規模と品目数を集計する。
    partner_totals = covered_market.groupby(
        ["refYear", "partnerCode", "partnerDesc"], as_index=False
    ).agg(japanx=(VALUE, "sum"), n_cmd=("cmdCode", "nunique"))
    yearly_japanx.append(partner_totals)
    print(
        f"{year}: 日本の(相手国,品目) {len(exported_commodities):,} 組"
        f" → 市場集約 {len(covered_market):,} 行 / 相手国 {len(partner_totals):,}"
    )

japanx = pd.concat(yearly_japanx, ignore_index=True)
japanx = japanx.rename(columns={"refYear": "year"})
japanx = japanx.sort_values(
    ["year", "japanx"], ascending=[True, False]
).reset_index(drop=True)

print(f"\n{len(japanx):,} 行（年 × 相手国）")
japanx.head(10)


### 検算

`japancountry ≤ japanx ≤ worldcountry` が全行で成り立つか確認する。
`japanx / worldcountry` は「日本が扱う品目が m の輸入市場のどれだけを覆っているか」、
`japancountry / japanx` は「その市場で日本が取れているシェア」と読める。

In [ ]:
chk3 = (japancountry
        .merge(japanx[["year", "partnerCode", "japanx", "n_cmd"]],
               on=["year", "partnerCode"], how="inner")
        .merge(worldcountry[["year", "partnerCode", "worldcountry"]],
               on=["year", "partnerCode"], how="left"))

eps = 1e-6
ng1 = (chk3["japancountry"] > chk3["japanx"] + eps).sum()
ng2 = (chk3["japanx"] > chk3["worldcountry"] + eps).sum()
print(f"japancountry > japanx    : {ng1}  ← 0 であること")
print(f"japanx > worldcountry    : {ng2}  ← 0 であること")
if ng1 or ng2:
    raise RuntimeError("大小関係が崩れている。集合の作り方を確認すること。")
print("大小関係 OK")

chk3["カバー率%"]  = (chk3["japanx"] / chk3["worldcountry"] * 100).round(2)
chk3["日本シェア%"] = (chk3["japancountry"] / chk3["japanx"] * 100).round(2)

print()
print("年ごとの集計（10億USD）:")
g = chk3.groupby("year").agg(japancountry=("japancountry", "sum"),
                             japanx=("japanx", "sum"),
                             worldcountry=("worldcountry", "sum"))
out = (g / 1e9).round(2)
out["カバー率%"]  = (g["japanx"] / g["worldcountry"] * 100).round(1)
out["日本シェア%"] = (g["japancountry"] / g["japanx"] * 100).round(2)
out

In [ ]:
# 主要国の内訳（直近年）
last = chk3["year"].max()
cols = ["partnerDesc", "n_cmd", "japancountry", "japanx", "worldcountry",
        "カバー率%", "日本シェア%"]
view = chk3[chk3["year"] == last].nlargest(10, "japancountry")[cols].copy()
for c in ["japancountry", "japanx", "worldcountry"]:
    view[c] = (view[c] / 1e6).round(1)     # 百万USD
print(f"{last}年 上位10か国（金額は百万USD、n_cmd は日本が輸出している品目数）")
view.reset_index(drop=True)

In [ ]:
# 保存（他の変数と同じ3形式）
VAR = "japanx"
tbl = japanx

d_ = OUT / VAR
d_.mkdir(parents=True, exist_ok=True)
for y, g in tbl.groupby("year"):
    (g.drop(columns="year").sort_values(VAR, ascending=False).reset_index(drop=True)
      .to_csv(d_ / f"{VAR}_{y}.csv", index=False, encoding="utf-8-sig"))
print(f"① 年ごと {len(list(d_.glob('*.csv')))} ファイル: {d_.resolve()}")

wide = tbl.pivot(index=["partnerCode", "partnerDesc"], columns="year", values=VAR).reset_index()
wide.columns.name = None
wide.to_csv(OUT / f"{VAR}_wide.csv", index=False, encoding="utf-8-sig")
print(f"② 横持ち: {VAR}_wide.csv ({len(wide)} か国)")

tbl.to_csv(OUT / f"{VAR}.csv", index=False, encoding="utf-8-sig")
print(f"③ 縦持ち: {VAR}.csv ({len(tbl):,} 行)")

wide[wide["partnerDesc"] == "China"]

# B. マージン分解 — EM / IM

前半で作った3変数から、比率の指標を2つ作る。

| 変数 | 定義 | 意味 |
|---|---|---|
| `EM` | `japanx / worldcountry` | **外延マージン。** 日本が扱う品目が、輸入国 *m* の輸入市場のどれだけを覆っているか |
| `IM` | `japancountry / japanx` | **内延マージン。** その覆っている市場の中で、日本が実際にどれだけ取れているか |

いずれも年 × 相手国。値域は 0〜1。

## 恒等式

定義から次が厳密に成り立つ。

```
EM × IM = (japanx / worldcountry) × (japancountry / japanx)
        = japancountry / worldcountry
        = 日本の総シェア
```

つまり **日本の市場シェアを「品目の広がり（EM）」と「品目内での取り分（IM）」に分解**している。
シェアが低い原因が「扱っている品目が狭いから」なのか「扱ってはいるが売れていないから」なのかを
切り分けられる。

**`share`（総シェア）は出力しない。** ただしこの恒等式は計算の正しさを確認する
強力な手段なので、検算の参照値としてのみ内部で使う。

## B-1. 前半の計算結果を引き継ぐ

CSVの再読み込みは行いません。


In [ ]:
# 前半で作成した表をそのまま利用する。
jc = japancountry
wc = worldcountry
jx = japanx


## B-2. 結合

`worldcountry` は日本が輸出していない国も含むため行数が多い（3,391 行）。
比率は日本の輸出がある組でしか定義できないので、`japancountry` を基準に内部結合する。

In [ ]:
KEYS = ["year", "partnerCode", "partnerDesc"]

df = (jc
      .merge(jx[["year", "partnerCode", "japanx", "n_cmd"]], on=["year", "partnerCode"], how="inner")
      .merge(wc[["year", "partnerCode", "worldcountry"]], on=["year", "partnerCode"], how="inner"))

print(f"結合後: {len(df):,} 行")

# 割り算の前に分母を検査する。0 や欠損があれば inf / NaN が混入する。
for col in ["japanx", "worldcountry"]:
    z, n = (df[col] == 0).sum(), df[col].isna().sum()
    print(f"  {col:13s} ゼロ {z} / 欠損 {n}")
    if z or n:
        raise RuntimeError(f"{col} に 0 または欠損がある。比率が定義できない。")

## B-3. EM / IM の計算

In [ ]:
df["EM"] = df["japanx"] / df["worldcountry"]        # 外延マージン
df["IM"] = df["japancountry"] / df["japanx"]        # 内延マージン

df = df[KEYS + ["n_cmd", "japancountry", "japanx", "worldcountry", "EM", "IM"]]
df = df.sort_values(["year", "japancountry"], ascending=[True, False]).reset_index(drop=True)

print(f"{len(df):,} 行")
df.head(10)

### 検算 — 恒等式

`EM × IM = share` が全行で成り立つか確認する。浮動小数点の丸めがあるため
厳密な等号ではなく相対誤差で判定する。

In [ ]:
import numpy as np

# 定義から EM × IM = japancountry / worldcountry（日本の総シェア）が厳密に成り立つ。
# share は出力しないが、検算の参照値としてここで作る。
ref = df["japancountry"] / df["worldcountry"]
lhs = df["EM"] * df["IM"]
ok = np.isclose(lhs, ref, rtol=1e-12, atol=0)

print(f"EM × IM = japancountry/worldcountry が成立: {ok.sum():,} / {len(df):,} 行")
print(f"最大の相対誤差: {(abs(lhs - ref) / ref).max():.2e}")
if not ok.all():
    raise RuntimeError("恒等式が崩れている。計算を確認すること。")

# 値域の検査。定義上 0 < EM ≤ 1、0 < IM ≤ 1 でなければならない。
for col in ["EM", "IM"]:
    lo, hi = df[col].min(), df[col].max()
    print(f"  {col}: 最小 {lo:.6f} / 最大 {hi:.6f}")
    if lo <= 0 or hi > 1 + 1e-9:
        raise RuntimeError(f"{col} が値域 (0, 1] を外れている。")
print("値域 OK")

## B-4. 全体の推移

In [ ]:
# 年ごとの集計値。個別国の平均ではなく、分子・分母をそれぞれ合計してから割る
# （加重平均。単純平均だと貿易額の小さい国の影響が過大になる）
agg = df.groupby("year").agg(japancountry=("japancountry", "sum"),
                            japanx=("japanx", "sum"),
                            worldcountry=("worldcountry", "sum"),
                            国数=("partnerCode", "nunique"))
agg["EM"] = (agg["japanx"] / agg["worldcountry"] * 100).round(1)
agg["IM"] = (agg["japancountry"] / agg["japanx"] * 100).round(2)

print("単位: EM / IM は %")
agg[["国数", "EM", "IM"]]

## B-5. 相手国別に見る

In [ ]:
last = df["year"].max()
view = df[df["year"] == last].nlargest(15, "japancountry").copy()
view["EM%"] = (view["EM"] * 100).round(1)
view["IM%"] = (view["IM"] * 100).round(2)
view["輸出額_百万USD"] = (view["japancountry"] / 1e6).round(1)

print(f"{last}年 日本の輸出額 上位15か国")
view[["partnerDesc", "n_cmd", "輸出額_百万USD", "EM%", "IM%"]].reset_index(drop=True)

### 読み方

`EM` と `IM` のどちらが低いかで、シェアが伸びない理由の解釈が変わる。

| パターン | 解釈 | 打ち手 |
|---|---|---|
| `EM` 低・`IM` 高 | 扱っている品目は強いが、市場の一部しか攻めていない | 品目を広げる |
| `EM` 高・`IM` 低 | 幅広く出しているが、どの品目でも競合に負けている | 競争力を上げる |
| 両方低い | 市場自体を取れていない | 参入戦略の見直し |

`n_cmd` が小さい国（品目数が数点）では `EM` が極端な値になりやすい。
2025年の実測では品目数の中央値が15、25パーセンタイルが3なので、
**半数近くの国は少数品目で構成されている**。回帰などに使う場合は `n_cmd` で足切りするか、
前半Aの `FIXED_BASKET = True` で品目集合を固定することを検討する。

In [ ]:
# EM と IM の分布（直近年）
d = df[df["year"] == last]
print(f"{last}年 {len(d)} か国  単位: %")
(d[["EM", "IM"]].describe()
   .loc[["min", "25%", "50%", "75%", "max"]] * 100).round(2)

## B-6. 国ごとの全年プール

年をまたいで1つの値にまとめたもの。**比率は分子・分母をそれぞれ対象年分合計してから割る**
（年ごとの EM を平均するのではない）。年ごとの比率を単純平均すると、貿易額の小さい年が
過大に効いてしまうため。

`n_cmd` は年ごとの値しか持っていないので、**全期間で通した品目数は元データから数え直す**。
「対象期間のどこかの年に日本が *m* へ輸出した HS6桁の種類数」であり、
年ごとの `n_cmd` の合計でも平均でもない。

In [ ]:
# 前半で読み込んだ日本の明細を再利用し、全期間の相手国×品目を数える。
partner_commodities = japan_partner_rows[["partnerCode", "cmdCode"]].drop_duplicates()
n_cmd_pooled = partner_commodities.groupby("partnerCode", as_index=False).size()
n_cmd_pooled = n_cmd_pooled.rename(columns={"size": "n_cmd_pooled"})
print(f"全期間で通した (相手国, 品目) の組: {n_cmd_pooled['n_cmd_pooled'].sum():,}")


In [ ]:
# 分子・分母を対象年分合計してから比率を出す
margins_all = (df.groupby(["partnerCode", "partnerDesc"], as_index=False)
                 .agg(years=("year", "nunique"),
                      japancountry=("japancountry", "sum"),
                      japanx=("japanx", "sum"),
                      worldcountry=("worldcountry", "sum"),
                      n_cmd_mean=("n_cmd", "mean")))

margins_all = margins_all.merge(n_cmd_pooled, on="partnerCode", how="left")
margins_all["n_cmd_mean"] = margins_all["n_cmd_mean"].round(1)

margins_all["EM"] = margins_all["japanx"] / margins_all["worldcountry"]
margins_all["IM"] = margins_all["japancountry"] / margins_all["japanx"]

margins_all = (margins_all[["partnerCode", "partnerDesc", "years",
                            "n_cmd_pooled", "n_cmd_mean",
                            "japancountry", "japanx", "worldcountry", "EM", "IM"]]
               .sort_values("japancountry", ascending=False)
               .reset_index(drop=True))

# 恒等式は集計後も成り立つ
assert np.isclose(margins_all["EM"] * margins_all["IM"],
                  margins_all["japancountry"] / margins_all["worldcountry"], rtol=1e-12).all()
print(f"{len(margins_all)} か国  （恒等式は集計後も成立）")

v = margins_all.head(15).copy()
v["輸出額_百万USD"] = (v["japancountry"] / 1e6).round(1)
v["EM%"] = (v["EM"] * 100).round(1)
v["IM%"] = (v["IM"] * 100).round(2)
v[["partnerDesc", "years", "n_cmd_pooled", "n_cmd_mean", "輸出額_百万USD", "EM%", "IM%"]]

### 年ごとと全年プールの違い

同じ国でも、年ごとの値を眺めるのと全期間で括るのとでは見え方が変わる。
下は中国の例。全年プールの EM は、年ごとの値の平均とは一致しない
（分母の大きい年に引っ張られるため）。

In [ ]:
CN = 156
per_year = df[df["partnerCode"] == CN][["year", "n_cmd", "EM", "IM"]].copy()
pooled = margins_all[margins_all["partnerCode"] == CN].iloc[0]

for c in ["EM", "IM"]:
    per_year[c] = (per_year[c] * 100).round(2)
print("年ごと（%）:")
print(per_year.to_string(index=False))
print()
print(f"年ごとの EM の単純平均 : {per_year['EM'].mean():.2f} %")
print(f"全年プールの EM        : {pooled['EM']*100:.2f} %   ← こちらを使う")
print()
print(f"品目数: 年ごと平均 {pooled['n_cmd_mean']:.1f} / 全期間で通すと {pooled['n_cmd_pooled']}")

## B-7. 保存

In [ ]:
# ① 年ごとに1ファイル
YEAR_DIR = OUT / "margins"
YEAR_DIR.mkdir(parents=True, exist_ok=True)
for y, g in df.groupby("year"):
    (g.drop(columns="year").reset_index(drop=True)
      .to_csv(YEAR_DIR / f"margins_{y}.csv", index=False, encoding="utf-8-sig"))
print(f"① 年ごと {len(list(YEAR_DIR.glob('*.csv')))} ファイル: {YEAR_DIR.resolve()}")

# ② 横持ち（EM と IM を別ファイルに）
for col in ["EM", "IM"]:
    w = df.pivot(index=["partnerCode", "partnerDesc"], columns="year", values=col).reset_index()
    w.columns.name = None
    w.to_csv(OUT / f"{col}_wide.csv", index=False, encoding="utf-8-sig")
    print(f"② 横持ち: {col}_wide.csv ({len(w)} か国)")

# ③ 縦持ち（3変数と比率をまとめた1枚）
df.to_csv(OUT / "margins.csv", index=False, encoding="utf-8-sig")
print(f"③ 縦持ち: margins.csv ({len(df):,} 行)")

# ④ 国ごとの全年プール
margins_all.to_csv(OUT / "margins_all_years.csv", index=False, encoding="utf-8-sig")
print(f"④ 全年プール: margins_all_years.csv ({len(margins_all)} か国)")

# ⑤ 年ごとの全国集計
agg.reset_index().to_csv(OUT / "margins_by_year.csv", index=False, encoding="utf-8-sig")
print(f"⑤ 年ごと集計: margins_by_year.csv ({len(agg)} 年)")

## メモ

- `EM` / `IM` は**比率なので、分母が小さい国では不安定になる**。
  `worldcountry` の最小値は 4.00 USD（実測）で、こうした極小市場では EM が 1 に張り付く。
- 年ごとの集計値は**加重平均**（分子・分母をそれぞれ合計してから割る）で出している。
  国ごとの EM を単純平均すると、貿易額の小さい国が過大に効いてしまう。
- `japanx` の品目集合が年ごとに変わるため、`EM` の時系列変化には
  「市場側の変化」と「日本の品目構成の変化」が混ざる。詳細は `DESIGN.md` を参照。

## 付録：変数を追加するときの雛形

同じ `japan_rows`（または `load_reporter()` で読んだ他国のデータ）から派生させる。

```python
# 例: 日本の品目別輸出額（年 × HS6桁）
japancmd = (japan_rows[japan_rows["partnerDesc"] == "World"]
            .groupby(["refYear", "cmdCode"], as_index=False)["primaryValue"]
            .sum()
            .rename(columns={"refYear": "year", "primaryValue": "japancmd"}))

# 例: 日本から相手国への品目別輸出額（年 × 相手国 × HS6桁）
japancountrycmd = (japan_partner_rows
                   .groupby(["refYear", "partnerCode", "cmdCode"], as_index=False)["primaryValue"]
                   .sum()
                   .rename(columns={"refYear": "year", "primaryValue": "japancountrycmd"}))
```

日本以外の報告国が必要なら `load_reporter(コード)` を使う。コードは
UN Comtradeの報告国参照表（`comtradeapicall.getReference("reporter")`）で調べられる。

### 注意点

- **`World` を混ぜない。** 個別相手国と合算すると必ず二重計上になる。
- **`cifvalue` は輸出データではほぼ空。** 金額は `primaryValue`（= FOB）を使う。
- **直近年は報告が出揃っていない。** 2024〜2025年は報告国数が少なく、
  時系列比較ではそのまま使うと過小評価になる。
- **HS 版が年で変わる。** 品目コード単位で年を跨いで追う場合、統廃合されたコードに注意
  （例: `010110` → `010121` / `010129`）。`classificationCode` で報告版を判別できる。